# A2 Usage segments

Users are clustered by how they use the feed: time of day, weekend share, video share, scroll depth and use of the swipe-down lists. The notebook checks how stable and how well separated the clusters are, whether they add anything to the activity score, and how they relate to next week's activity. Outputs: `20_*`.

## 1. Setup and usage mix

In [1]:
# Environment setup: libraries and shared project code
import hashlib, json, warnings
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score, roc_auc_score

from thesis.config import P, O, TAG, SEED, connect
from thesis.panel import load, first_origin_tiers
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore", category=RuntimeWarning)   # spurious matmul warnings of NumPy on macOS

In [2]:
# Usage mix of every user with 20+ cards shown on days 1-15
df, FEATS, tr, va, te = load()
users = df[["userId", "bucket"]].drop_duplicates("userId")
con = connect()
con.register("users", users)
con.execute(f"""CREATE TABLE imp AS
  SELECT i.userId, u.bucket, CAST(i.dt AS INT) AS dt, i.isClick, i.has_detail, i.impressPosition, m.type AS card_type,
         CAST(((i.impressTime / 1000 + 8 * 3600) % 86400) / 3600 AS INT) AS hour,
         dayofweek(DATE '2019-10-31' + CAST(i.dt AS INT)) IN (0, 6) AS weekend
  FROM read_parquet('{P}/impressions_place/d*.parquet') i JOIN users u USING (userId)
  LEFT JOIN '{P}/mlog_demographics.parquet' m USING (mlogId)
  WHERE CAST(i.dt AS INT) <= 15""")
X = con.execute("""SELECT userId, any_value(bucket) AS bucket, count(*) AS impr, count(DISTINCT dt) AS visit_days,
    avg((card_type = '2')::INT) AS video_share,
    avg((hour < 6)::INT) AS night, avg((hour >= 6 AND hour < 12)::INT) AS morning,
    avg((hour >= 12 AND hour < 18)::INT) AS afternoon, avg((hour >= 18)::INT) AS evening,
    avg(weekend::INT) AS weekend_share, ln(1 + avg(impressPosition)) AS log_mean_position,
    avg(has_detail::INT) AS swipe_down_share
  FROM imp GROUP BY userId HAVING count(*) >= 20 ORDER BY userId""").df()
con.close()

## 2. Clusters and their stability

In [3]:
# k-means for k = 2..8 with silhouette and bootstrap stability; choose the best stable solution
MIX = ["video_share", "night", "morning", "afternoon", "evening", "weekend_share", "log_mean_position",
       "swipe_down_share"]
trn = X[X.bucket < 70].reset_index(drop=True)
tst = X[X.bucket >= 85].reset_index(drop=True)
mu, sd = trn[MIX].mean(), trn[MIX].std()
Z, Zt = ((trn[MIX] - mu) / sd).values, ((tst[MIX] - mu) / sd).values
rng = np.random.default_rng(SEED)
sub = rng.choice(len(Z), min(20000, len(Z)), replace=False)          # subsample for the silhouette

res = {"users_train": len(trn), "users_test": len(tst), "features": MIX, "by_k": {}}
fits = {}
for k in range(2, 9):
    km = KMeans(k, n_init=10, random_state=SEED).fit(Z)
    fits[k] = km
    ari = [adjusted_rand_score(km.labels_, KMeans(k, n_init=10, random_state=SEED + b).fit(Z[idx]).predict(Z))
           for b, idx in enumerate(rng.integers(0, len(Z), (20, len(Z))))]
    res["by_k"][k] = {"silhouette": round(float(silhouette_score(Z[sub], km.labels_[sub])), 4),
                      "stability_ari_mean": round(float(np.mean(ari)), 4),
                      "stability_ari_min": round(float(np.min(ari)), 4),
                      "cluster_shares": np.round(np.bincount(km.labels_) / len(Z), 3).tolist()}

stable = [k for k in res["by_k"] if res["by_k"][k]["stability_ari_min"] >= 0.8]
best = max(stable, key=lambda k: res["by_k"][k]["silhouette"])
res["k_chosen"] = best
res["k_stable"] = stable
km = fits[best]
trn["cluster"] = km.labels_
tst["cluster"] = km.predict(Zt)
profiles = trn.groupby("cluster")[MIX + ["impr"]].mean().assign(users=trn.cluster.value_counts().sort_index()).round(4)
profiles.to_csv(O / f"20_cluster_profiles{TAG}.csv")
pd.DataFrame(res["by_k"]).T.assign(chosen=lambda d: d.index == best)

,silhouette,stability_ari_mean,stability_ari_min,cluster_shares,chosen
2,0.1477,0.9896,0.9779,"[0.613, 0.387]",False
3,0.1755,0.7254,0.5188,"[0.338, 0.398, 0.265]",False
4,0.2089,0.982,0.9473,"[0.28, 0.411, 0.203, 0.105]",False
5,0.2048,0.9788,0.9378,"[0.083, 0.225, 0.162, 0.119, 0.411]",False
6,0.2128,0.9915,0.9821,"[0.402, 0.08, 0.054, 0.2, 0.114, 0.15]",True
7,0.2163,0.8734,0.5398,"[0.388, 0.064, 0.05, 0.112, 0.162, 0.079, 0.145]",False
8,0.1776,0.8762,0.5153,"[0.255, 0.056, 0.113, 0.116, 0.108, 0.224, 0.0...",False


In [4]:
# Profile of each cluster (training users)
profiles

,video_share,night,morning,afternoon,evening,weekend_share,log_mean_position,swipe_down_share,impr,users
cluster,,,,,,,,,,
0,0.3558,0.0783,0.1774,0.2910,0.4533,0.2646,1.0692,0.0026,55.4160,78746
1,0.3739,0.7067,0.0527,0.0945,0.1461,0.3112,2.0417,0.0105,73.2390,15707
2,0.6123,0.0664,0.1449,0.2869,0.5017,0.3719,2.0949,0.0975,111.7809,10548
3,0.3787,0.0198,0.0338,0.0559,0.8905,0.2795,2.5357,0.0107,101.8765,39250
4,0.3863,0.0243,0.7531,0.1018,0.1208,0.3394,2.1668,0.0101,74.8212,22362
5,0.3776,0.0232,0.0537,0.8176,0.1056,0.3646,2.3790,0.0111,87.9643,29429


## 3. Value beyond the activity score

In [5]:
# Binomial models with and without cluster dummies on top of the score, five-fold cross-fitting grouped by user
def glm(Xd, k, n, iters=50):
    """Binomial GLM by Newton steps."""
    th = np.zeros(Xd.shape[1])
    for _ in range(iters):
        p = 1 / (1 + np.exp(-(Xd @ th)))
        step = np.linalg.solve(Xd.T @ (Xd * (n * p * (1 - p))[:, None]) + 1e-8 * np.eye(len(th)), Xd.T @ (k - n * p))
        th += step
        if np.abs(step).max() < 1e-9:
            break
    return th


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


S = (pd.read_parquet(O / f"19_scores{TAG}.parquet")
     .merge(te[["userId", "origin_day", "y_visit_days", "y_clicks", "y_impr"]], on=["userId", "origin_day"])
     .merge(tst[["userId", "cluster"]], on="userId"))
fold = S.userId.map(lambda u: int(hashlib.md5(("f" + u).encode()).hexdigest()[:8], 16) % 5).values  # deterministic
D = pd.get_dummies(S.cluster, prefix="c", drop_first=True).astype(float).values
added = {}
for name, score, k, n, sel in (("return", S.p_return, (S.y_visit_days > 0).astype(float), np.ones(len(S)),
                                np.ones(len(S), bool)),
                               ("clicks", S.p_click, S.y_clicks.astype(float), S.y_impr.astype(float),
                                (S.y_impr > 0).values)):
    z = logit(score.values)[sel]
    kk, nn, dd, ff = k.values[sel], np.asarray(n)[sel], D[sel], fold[sel]
    out = {m: np.zeros(sel.sum()) for m in ("score_only", "score_plus_cluster")}
    for f in range(5):
        a, b = ff != f, ff == f
        for m, Xd in (("score_only", np.column_stack([np.ones(len(z)), z])),
                      ("score_plus_cluster", np.column_stack([np.ones(len(z)), z, dd]))):
            out[m][b] = 1 / (1 + np.exp(-(Xd[b] @ glm(Xd[a], kk[a], nn[a]))))
    ll = {m: float(-(kk * np.log(np.clip(q, 1e-6, 1)) + (nn - kk) * np.log(np.clip(1 - q, 1e-6, 1))).sum() / nn.sum())
          for m, q in out.items()}
    yb = kk > 0
    hb = np.ones(len(kk), bool) if name == "return" else nn >= 20
    added[name] = {"rows": int(sel.sum()), "logloss": {m: round(v, 5) for m, v in ll.items()},
                   "auc": {m: round(float(roc_auc_score(yb[hb], q[hb])), 4) for m, q in out.items()}}
res["added_value_beyond_score"] = added
(O / f"20_segments{TAG}.json").write_text(json.dumps(res, indent=2))
pd.concat({t: pd.DataFrame({k: v for k, v in added[t].items() if isinstance(v, dict)}) for t in added})

logloss     auc
return score_only          0.29921  0.8324
       score_plus_cluster  0.29924  0.8324
clicks score_only          0.19298  0.8315
       score_plus_cluster  0.19299  0.8315

## 4. Clusters by next week's activity group

In [6]:
# Cluster by next week's group (1 no return, 2 returned without a click, 3 one or two clicks, 4 three or more)
Sc = pd.read_parquet(O / f"19_scores{TAG}.parquet")
tiers = (first_origin_tiers(Sc)[["userId", "origin_day"]].merge(te[["userId", "origin_day", "y_visit_days", "y_clicks"]],
                                                                on=["userId", "origin_day"])
         .merge(tst[["userId", "cluster"]], on="userId"))
tiers["group"] = np.select([tiers.y_visit_days == 0, tiers.y_clicks == 0, tiers.y_clicks <= 2], [1, 2, 3], 4)
ct = pd.crosstab(tiers.cluster, tiers.group)
chi2, p_, dof, _ = chi2_contingency(ct)
res["clusters_by_group"] = {"users": int(len(tiers)), "chi2": round(float(chi2), 1), "dof": int(dof), "p_value": float(p_),
                           "cramers_v": round(float(np.sqrt(chi2 / (len(tiers) * (min(ct.shape) - 1)))), 4)}
(ct / ct.sum(axis=0)).round(4).to_csv(O / f"20_clusters_by_group{TAG}.csv")      # share of each group in each cluster
(ct / ct.sum(axis=0)).round(3)

group,1,2,3,4
cluster,,,,
0,0.342,0.597,0.450,0.241
1,0.089,0.069,0.073,0.072
2,0.054,0.018,0.081,0.184
3,0.221,0.129,0.170,0.245
4,0.116,0.088,0.099,0.098
5,0.177,0.099,0.128,0.160


## 5. Clusters among users seen on at least eight days

In [7]:
# k-means for k = 2..8 among users with 8+ visit days on days 1-15 (training users), standardized as before
trn8 = trn[trn.visit_days >= 8].reset_index(drop=True)
Z8 = ((trn8[MIX] - mu) / sd).values
sub8 = rng.choice(len(Z8), min(20000, len(Z8)), replace=False)
res["eight_visit_days"] = {"users_train": len(trn8), "by_k": {}}
for k in range(2, 9):
    km8 = KMeans(k, n_init=10, random_state=SEED).fit(Z8)
    ari = [adjusted_rand_score(km8.labels_, KMeans(k, n_init=10, random_state=SEED + b).fit(Z8[idx]).predict(Z8))
           for b, idx in enumerate(rng.integers(0, len(Z8), (20, len(Z8))))]
    res["eight_visit_days"]["by_k"][k] = {
        "silhouette": round(float(silhouette_score(Z8[sub8], km8.labels_[sub8])), 4),
        "stability_ari_min": round(float(np.min(ari)), 4),
        "cluster_shares": np.round(np.bincount(km8.labels_) / len(Z8), 3).tolist()}
    if k == best:
        res["eight_visit_days"]["profiles_at_k_chosen"] = (
            trn8.assign(cluster=km8.labels_).groupby("cluster")[MIX].mean().round(4).to_dict("index"))
(O / f"20_segments{TAG}.json").write_text(json.dumps(res, indent=2))
pd.DataFrame(res["eight_visit_days"]["by_k"]).T

,silhouette,stability_ari_min,cluster_shares
2,0.3311,0.9716,"[0.768, 0.232]"
3,0.3004,0.952,"[0.709, 0.123, 0.168]"
4,0.2213,0.4133,"[0.602, 0.11, 0.133, 0.155]"
5,0.1655,0.3976,"[0.351, 0.093, 0.294, 0.141, 0.121]"
6,0.17,0.934,"[0.128, 0.061, 0.11, 0.299, 0.153, 0.25]"
7,0.1689,0.9382,"[0.209, 0.101, 0.058, 0.167, 0.211, 0.138, 0.115]"
8,0.1676,0.9283,"[0.098, 0.195, 0.074, 0.05, 0.157, 0.087, 0.20..."


## 6. Summary
Stable but weakly separated clusters that add nothing to the activity score.